# Лабораторная работа №1

Решите следующие задачи для данных велопарковок Сан-Франциско (trips.csv, stations.csv):
* Найти велосипед с максимальным временем пробега.
* Найти наибольшее геодезическое расстояние между станциями.
* Найти путь велосипеда с максимальным временем пробега через станции.
* Найти количество велосипедов в системе.
* Найти пользователей потративших на поездки более 3 часов.

## 1. Подключение Spark, загрузка и чтение датасетов.

In [24]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Lab1") \
    .getOrCreate()
print("Spark session created: ")
spark

Spark session created: 


In [25]:
# Загружаем поездки
trips_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("timestampFormat", "M/d/y H:m") \
    .csv("trip.csv")

# Загружаем станции
stations_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("timestampFormat", "M/d/y") \
    .csv("station.csv")

print("Данные загружены:")
print(f"Поездок: {trips_df.count()}, столбцов: {len(trips_df.columns)}")
print(f"Станций: {stations_df.count()}, столбцов: {len(stations_df.columns)}")

Данные загружены:
Поездок: 669959, столбцов: 11
Станций: 70, столбцов: 7


In [26]:
print("\nПервые 5 поездок:")
trips_df.show(5)

print("\nПервые 5 станций:")
stations_df.show(5)


Первые 5 поездок:
+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|  id|duration|         start_date|  start_station_name|start_station_id|           end_date|    end_station_name|end_station_id|bike_id|subscription_type|zip_code|
+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|4576|      63|2013-08-29 14:13:00|South Van Ness at...|              66|2013-08-29 14:14:00|South Van Ness at...|            66|    520|       Subscriber|   94127|
|4607|      70|2013-08-29 14:42:00|  San Jose City Hall|              10|2013-08-29 14:43:00|  San Jose City Hall|            10|    661|       Subscriber|   95138|
|4130|      71|2013-08-29 10:16:00|Mountain View Cit...|              27|2013-08-29 10:17:00|Mountain View Cit...|            27|     48|       Subscriber| 

## 2. Решение задач.

### 2.1 Найти велосипед с максимальным временем пробега.

In [27]:
max_duration_bike = (trips_df.groupBy("bike_id").agg(F.sum("duration").alias("total_duration"))).orderBy(F.desc("total_duration"))
max_bike_id = max_duration_bike.collect()[0]["bike_id"]

max_duration_bike.show(1)

+-------+--------------+
|bike_id|total_duration|
+-------+--------------+
|    535|      18611693|
+-------+--------------+
only showing top 1 row


### 2.2 Найти наибольшее геодезическое расстояние между станциями.

In [28]:
station_coords = stations_df.select("id", "lat", "long")
cross_joined = station_coords.alias("s1").crossJoin(station_coords.alias("s2")).filter(F.col("s1.id") < F.col("s2.id"))

stations_with_distance = cross_joined.withColumn(
    "distance",
    F.lit(6371) * F.acos(
        F.sin(F.radians(F.col("s1.lat"))) * F.sin(F.radians(F.col("s2.lat"))) +
        F.cos(F.radians(F.col("s1.lat"))) * F.cos(F.radians(F.col("s2.lat"))) *
        F.cos(F.radians(F.col("s2.long")) - F.radians(F.col("s1.long")))
    )
)

max_distance = stations_with_distance.orderBy(F.desc("distance")).limit(1)
max_distance.select(
    F.col("s1.id").alias("station1_id"),
    F.col("s2.id").alias("station2_id"),
    F.col("distance").alias("max_distance_km")
).show()

+-----------+-----------+-----------------+
|station1_id|station2_id|  max_distance_km|
+-----------+-----------+-----------------+
|         16|         60|69.92087595421542|
+-----------+-----------+-----------------+



### 2.3 Найти путь велосипеда с максимальным временем пробега через станции.


In [29]:
path_bike_max = trips_df.select("id","bike_id","start_station_name", "end_station_name") \
    .filter(F.col("bike_id") == max_bike_id) \
    .orderBy(F.col("id").cast(T.IntegerType()))

path_bike_max.show(path_bike_max.count(), truncate=False)

+------+-------+---------------------------------------------+---------------------------------------------+
|id    |bike_id|start_station_name                           |end_station_name                             |
+------+-------+---------------------------------------------+---------------------------------------------+
|4966  |535    |Post at Kearney                              |San Francisco Caltrain (Townsend at 4th)     |
|5067  |535    |San Francisco Caltrain (Townsend at 4th)     |San Francisco Caltrain 2 (330 Townsend)      |
|5179  |535    |San Francisco Caltrain 2 (330 Townsend)      |Market at Sansome                            |
|5199  |535    |Market at Sansome                            |2nd at South Park                            |
|7806  |535    |2nd at Townsend                              |Davis at Jackson                             |
|11422 |535    |San Francisco City Hall                      |Civic Center BART (7th at Market)            |
|12245 |535    |Civ

### 2.4 Найти количество велосипедов в системе.


In [30]:
bike_count = trips_df.select("bike_id").distinct().count()

print(f"Number of bikes in the system: {bike_count}")

Number of bikes in the system: 700


### 2.5 Найти пользователей потративших на поездки более 3 часов.

In [31]:
users_time = trips_df.groupBy("zip_code") \
    .agg((F.sum("duration")/3600).alias("total_hours")) \
    .filter(F.col("total_hours") > 3) \
    .orderBy(F.desc("total_hours"))
users_time.show()

+--------+------------------+
|zip_code|       total_hours|
+--------+------------------+
|   94107|13821.433888888889|
|     nil|12701.541666666666|
|    NULL| 7700.909166666666|
|   94105| 7110.035555555555|
|   94133| 6010.465277777777|
|   94102| 5313.339166666667|
|   94103| 5313.163333333333|
|   95531| 4797.333333333333|
|   94111|3956.9436111111113|
|   95112|3539.5472222222224|
|   94109| 3349.202222222222|
|   94040|2168.8683333333333|
|   94110| 2061.648888888889|
|   94117|1917.0313888888888|
|   94301|1830.6605555555554|
|   94041|1743.4122222222222|
|   94158|1735.6019444444444|
|   94306|1541.8452777777777|
|   94025|1438.3991666666666|
|   94108|1424.3227777777777|
+--------+------------------+
only showing top 20 rows
